# 고객 행동 분석

`customer_behavior_analysis.py`의 분석 흐름을 주피터 노트북으로 옮긴 파일입니다.

확인할 내용:

- 고객별 총소비금액, 평균소비금액, 거래횟수
- 고객별 사기거래횟수와 사기비율
- 상위 소비 고객 TOP 10
- 거래횟수와 평균소비금액의 관계
- 시간대별 평균 소비금액
- 사기 거래와 정상 거래의 평균 금액 차이

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from matplotlib import font_manager

In [ ]:
# 한글 폰트 설정
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

display(HTML("""
<style>
.jp-Notebook { font-size: 13px; }
.jp-RenderedMarkdown { font-size: 13px; line-height: 1.55; }
.jp-RenderedMarkdown h1 { font-size: 22px; }
.jp-RenderedMarkdown h2 { font-size: 17px; }
.jp-RenderedMarkdown h3 { font-size: 15px; }
.jp-OutputArea-output, .jp-RenderedText, .jp-OutputArea pre { font-size: 12px; }
.dataframe { font-size: 11px; }
</style>
"""))

## 1. 데이터 로드

원본 파이썬 파일은 `data/credit_card_transactions.csv`를 읽습니다. 노트북에서는 현재 프로젝트 상태에 맞게 `data/`가 없으면 `step1/data/`를 자동으로 사용합니다.

In [ ]:
def get_data_path():
    current_dir = Path.cwd()
    candidates = [
        current_dir / "data" / "credit_card_transactions.csv",
        current_dir / "step1" / "data" / "credit_card_transactions.csv",
        current_dir.parent / "data" / "credit_card_transactions.csv",
        current_dir.parent / "step1" / "data" / "credit_card_transactions.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "credit_card_transactions.csv 파일을 찾을 수 없습니다. "
        "data/ 또는 step1/data/ 폴더를 확인하세요."
    )


data_path = get_data_path()
print(f"[데이터 파일] {data_path}")

df = pd.read_csv(data_path)

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

required_columns = {"cc_num", "amt", "is_fraud", "trans_date_trans_time"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")

df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day"] = df["trans_date_trans_time"].dt.day_name()

print(df.shape)
df.head()

## 2. 고객 단위 행동 분석

`cc_num` 기준으로 고객을 묶고, 총소비금액, 평균소비금액, 거래횟수, 사기거래횟수, 사기비율을 계산합니다.

In [ ]:
customer_group = df.groupby("cc_num")

customer_behavior = customer_group.agg(
    총소비금액=("amt", "sum"),
    평균소비금액=("amt", "mean"),
    거래횟수=("amt", "count"),
    사기거래횟수=("is_fraud", "sum"),
)

customer_behavior["사기비율"] = (
    customer_behavior["사기거래횟수"] / customer_behavior["거래횟수"]
)

customer_behavior.head()

## 3. 상위 소비 고객 TOP 10

총소비금액이 큰 고객 10명을 확인합니다.

In [ ]:
top_spenders = customer_behavior.sort_values("총소비금액", ascending=False).head(10)
top_spenders

In [ ]:
plt.figure(figsize=(9, 4))
top_spenders["총소비금액"].plot(kind="bar")
plt.title("상위 10 고객 - 총 소비금액")
plt.xlabel("고객 (cc_num)")
plt.ylabel("총 소비금액")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. 거래 횟수 vs 평균 소비금액

거래를 많이 하는 고객과 한 번에 큰 금액을 쓰는 고객을 구분해서 볼 수 있습니다.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
    customer_behavior["거래횟수"],
    customer_behavior["평균소비금액"],
    alpha=0.6,
)
plt.title("거래 횟수 vs 평균 소비금액")
plt.xlabel("거래 횟수")
plt.ylabel("평균 소비금액")
plt.tight_layout()
plt.show()

## 5. 시간대별 평균 소비금액

거래 발생 시간에서 `hour`를 추출해 시간대별 평균 결제 금액을 확인합니다.

In [ ]:
hourly_spend = df.groupby("hour")["amt"].mean()

plt.figure(figsize=(7, 4))
hourly_spend.plot(kind="line", marker="o")
plt.title("시간대별 평균 소비금액")
plt.xlabel("시간 (Hour)")
plt.ylabel("평균 금액")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 6. 사기 vs 정상 거래 평균 금액 비교

`is_fraud` 값이 0인 정상 거래와 1인 사기 거래의 평균 결제 금액을 비교합니다.

In [ ]:
fraud_group = df.groupby("is_fraud")["amt"].mean()

plt.figure(figsize=(5, 4))
fraud_group.plot(kind="bar")
plt.title("사기 vs 정상 거래 - 평균 거래 금액")
plt.xlabel("사기 여부 (0=정상, 1=사기)")
plt.ylabel("평균 거래 금액")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. 결과 해석

- `총소비금액`이 큰 고객은 전체 소비 규모가 큰 고객입니다.
- `평균소비금액`이 큰 고객은 1회 결제 금액이 큰 고객입니다.
- `거래횟수`가 큰 고객은 활동성이 높은 고객입니다.
- `사기비율`이 높은 고객은 위험 계정 후보로 볼 수 있습니다.
- 시간대별 평균 소비금액이 특정 시간에 튀면 이상 거래 탐지 feature로 활용할 수 있습니다.